# Lab 1: 3DGS Attribute Exploration

**Objectives:**
- Parse the full 59-parameter layout of a 3D Gaussian splat
- Implement quaternion-to-rotation-matrix conversion from scratch
- Reconstruct covariance matrices Σ = RSS^T R^T and verify PSD
- Visualize attribute distributions and compute uncompressed storage cost


In [ ]:
!pip install -q numpy matplotlib scipy
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
print('Ready.')

## Section 1: Generating Synthetic 3DGS Data

We generate N=50,000 Gaussians with realistic attribute distributions matching what you'd see in a trained 3DGS scene.

In [ ]:
np.random.seed(42)
N = 50_000

# Positions: clustered around origin with some structure
positions = np.random.randn(N, 3) * 2.0

# Scales: log-normal (small to medium Gaussians)
log_scales = np.random.randn(N, 3) * 0.5 - 2.0
scales = np.exp(log_scales)  # all positive

# Rotations: random unit quaternions (w, x, y, z)
quats_raw = np.random.randn(N, 4)
quats = quats_raw / np.linalg.norm(quats_raw, axis=1, keepdims=True)

# Opacity: pre-sigmoid logits, biased toward visible (high opacity)
opacity_logits = np.random.randn(N) * 2.0 + 1.0
opacities = 1.0 / (1.0 + np.exp(-opacity_logits))  # sigmoid

# DC color (SH order 0): RGB in [0,1]
sh_dc = np.random.rand(N, 3)

# SH AC (orders 1-3): 45 coefficients, small magnitudes
sh_ac = np.random.randn(N, 45) * 0.1

total_params = 3 + 3 + 4 + 1 + 3 + 45
total_bytes = N * total_params * 4
print(f'Parameters per Gaussian: {total_params}')
print(f'N = {N:,} Gaussians')
print(f'Uncompressed size: {N} x {total_params} x 4 bytes = {total_bytes/1e6:.1f} MB')
print(f'SH fraction: {N*45*4/total_bytes*100:.1f}% of total storage')

## Section 2: Quaternion-to-Rotation-Matrix

We implement the standard formula from scratch and verify orthogonality.

In [ ]:
def quat_to_rotmat(q):
    """Convert unit quaternion (w,x,y,z) to 3x3 rotation matrix.
    q: shape (..., 4)
    Returns: shape (..., 3, 3)
    """
    w, x, y, z = q[...,0], q[...,1], q[...,2], q[...,3]
    R = np.stack([
        np.stack([1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)], axis=-1),
        np.stack([  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)], axis=-1),
        np.stack([  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)], axis=-1),
    ], axis=-2)
    return R

# Verify: 90 deg rotation around Z-axis
angle = np.pi / 2
q_test = np.array([np.cos(angle/2), 0, 0, np.sin(angle/2)])
R_test = quat_to_rotmat(q_test)
print('90° around Z:')
print(np.round(R_test, 4))
print('Expected: [[0,-1,0],[1,0,0],[0,0,1]]')

# Verify orthogonality on batch
R_batch = quat_to_rotmat(quats[:500])
I_batch = np.eye(3)[np.newaxis].repeat(500, axis=0)
ortho_err = np.max(np.abs(R_batch @ R_batch.transpose(0,2,1) - I_batch))
print(f'\nMax orthogonality error (500 random quats): {ortho_err:.2e}')

## Section 3: Covariance Reconstruction — Σ = RSS^T R^T

In [ ]:
def build_covariance(scales, quats):
    """Build 3x3 covariance matrices Sigma = R S S^T R^T"""
    N = scales.shape[0]
    R = quat_to_rotmat(quats)  # (N,3,3)
    S = np.zeros((N, 3, 3))
    for i in range(3):
        S[:, i, i] = scales[:, i]
    Sigma = R @ (S @ S.transpose(0,2,1)) @ R.transpose(0,2,1)
    return Sigma

Sigma = build_covariance(scales[:1000], quats[:1000])
print(f'Covariance shape: {Sigma.shape}')
print(f'First covariance matrix:\n{Sigma[0].round(5)}')

# Verify PSD: all eigenvalues >= 0
eigvals = np.linalg.eigvalsh(Sigma)
print(f'\nMin eigenvalue: {eigvals.min():.2e}  (should be >= 0)')
print(f'All PSD: {(eigvals >= -1e-10).all()}')

# Gaussian volume = sqrt(det(Sigma))
dets = np.linalg.det(Sigma)
volumes = np.sqrt(np.abs(dets))
print(f'\nVolume  mean={volumes.mean():.4f}  median={np.median(volumes):.6f}  max={volumes.max():.4f}')

## Section 4: Attribute Distribution Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Opacity
axes[0,0].hist(opacities, bins=60, color='steelblue', edgecolor='none', alpha=0.85)
axes[0,0].axvline(0.05, color='red', linestyle='--', label='prune threshold')
axes[0,0].set_title('Opacity Distribution'); axes[0,0].set_xlabel('Opacity'); axes[0,0].legend()

# Max scale (log10)
axes[0,1].hist(np.log10(scales.max(axis=1)), bins=60, color='coral', alpha=0.85)
axes[0,1].set_title('Max Scale (log10)'); axes[0,1].set_xlabel('log10(max scale)')

# Position scatter colored by opacity
idx = np.random.choice(N, 3000)
sc = axes[0,2].scatter(positions[idx,0], positions[idx,1],
                       c=opacities[idx], cmap='viridis', s=1.5, alpha=0.7)
plt.colorbar(sc, ax=axes[0,2], label='Opacity')
axes[0,2].set_title('Positions (x,y) colored by opacity')

# DC color per channel
for i, (ch, col) in enumerate(zip('RGB', ['red','green','blue'])):
    axes[1,0].hist(sh_dc[:,i], bins=50, alpha=0.5, color=col, label=ch)
axes[1,0].set_title('DC Color (SH order 0)'); axes[1,0].legend()

# SH AC magnitude by order
labels = ['Order 1 (9 params)', 'Order 2 (15 params)', 'Order 3 (21 params)']
slices = [(0,9),(9,24),(24,45)]
means = [np.abs(sh_ac[:,s:e]).mean() for s,e in slices]
bars = axes[1,1].bar(labels, means, color=['#4f6ef7','#22c55e','#f59e0b'])
axes[1,1].set_title('Mean |SH AC| by Harmonic Order'); axes[1,1].tick_params(axis='x', labelsize=8)

# Quaternion components
for i, (comp, col) in enumerate(zip('wxyz', ['purple','blue','green','red'])):
    axes[1,2].hist(quats[:,i], bins=50, alpha=0.5, color=col, label=comp)
axes[1,2].set_title('Quaternion Components'); axes[1,2].legend()

plt.tight_layout()
plt.show()

## Section 5: Storage Cost Breakdown

In [ ]:
groups = [
    ('Position (xyz)',      3),
    ('Scale (s1,s2,s3)',    3),
    ('Rotation (quat)',     4),
    ('Opacity',             1),
    ('DC Color (SH-0)',     3),
    ('SH AC (orders 1-3)', 45),
]
print(f'{"Attribute":<25} {"Params":>7} {"Size (MB)":>12} {"% total":>10}')
print('-' * 58)
total = 0
for name, p in groups:
    mb = N * p * 4 / 1e6
    total += mb
    print(f'{name:<25} {p:>7} {mb:>11.2f}  {mb/(N*59*4/1e6)*100:>8.1f}%')
print('-' * 58)
print(f'{"TOTAL":<25} {59:>7} {total:>11.2f}')
print(f'\nSH AC dominates: {N*45*4/1e6/total*100:.1f}% of storage')
print(f'\nCompression targets:')
print(f'  5.94x (quant only):  {total/5.94:.1f} MB')
print(f'  22x   (SOG):         {total/22:.1f} MB')

## Key Takeaways

- A 3DGS scene has **59 float32 parameters per Gaussian** → ~236 bytes each
- **SH AC coefficients (45 params) account for ~76%** of storage — the highest-leverage compression target
- The covariance factorization **Σ = RSS^T R^T** guarantees positive semi-definiteness by construction
- Opacity distribution is biased toward high values (visible surfaces) → pruning low-opacity Gaussians is cheap
- Scale distribution is log-normal → quantizing in log space is more efficient than linear quantization
